---
title: 1a) The Connectome Interpreter Tutorial - Linear Methods (Effective Connectivity)
authors:
  - name: Aarushi Vardhan
    affiliations:
      - University of Toronto / University of Cambridge
  - name: Sapolnach Prompiengchai
    affiliations:
      - University of Oxford
---

### 1. Install Connectome Interpreter

Before starting, please install the **Connectome Interpreter** package and download the connectivity data used in this tutorial. Open your terminal and activate the environment you will use for this tutorial. Then run:

```bash
pip install git+https://github.com/YijieYin/connectome_interpreter.git
```

This tutorial is adapted from the [Linear Methods tutorial notebook](https://github.com/YijieYin/connectome_interpreter) of The Connectome Interpreter.

In [4]:
# 1. load your packages at the top
import pandas as pd
project_dir = '/Users/aarushivardhan/Desktop/MB'
import plotly.express as px
import os
from tqdm import tqdm
import scipy.sparse as sp
from connectome_interpreter import *
from dotenv import load_dotenv
from neuprint import Client
load_dotenv()
c = Client(os.getenv("NEUPRINT_SERVER"), dataset=os.getenv("NEUPRINT_DATASET"), token=os.getenv("NEUPRINT_TOKEN"))


# Get cell types function list. 
sheet_id = "1VHCEnurOdb4FDC_NUKZX_BpBckQ9LpKxv0CsK_ObVok"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid=0"
df = pd.read_csv(url)
type_to_function = dict(zip(df["cell_type"], df["known_function"])) # type_to_function contains a list of cell types and their experimentally-tested functions. 

# What is effective connectivity?

So far, we have looked at **direct connections** between neurons. For example, suppose neuron A connects directly to neuron C:

$$
A \longrightarrow C
$$

We can describe the strength of this connection as the fraction of C's input that comes from A. For example, if A provides 60% of C's input, the connection strength is

$$
A\rightarrow C = 0.60.
$$

We looked at this type of normalisation in [Notebook 1C of the neuPrint tutorial](./6c-neuprint_plotting.ipynb).

But neurons do not only influence neurons to which they are directly connected. Activity can also travel through **intermediate neurons**.

For example:

$$
A \longrightarrow B \longrightarrow C
$$

Here, A does not connect directly to C. Nevertheless, A can influence C through B.

This is the basic idea behind **effective connectivity**: we want to measure how strongly one neuron can influence another through indirect, multi-step paths.

## Following one indirect path

Consider the following pathway:

$$
A \xrightarrow{40\%} B \xrightarrow{60\%} C
$$

The percentages tell us how much of the **target neuron's input** comes from the source neuron.

So:

* A provides 40% of B's input.
* B provides 60% of C's input.

We want to know:

> How much of C's input can be attributed to A through the path {math}`A \rightarrow B \rightarrow C`?

We multiply the two fractions:

$$
0.40\times0.60=0.24.
$$

So the effective connection from A to C **along this path** is

$$
A\rightarrow B\rightarrow C = 0.24.
$$

In other words, A accounts for 24% of C's input through this particular two-step path.

### Why do we multiply?

Imagine C receives 100 units of input.

B provides 60% of C's input:

$$
100\times0.60=60.
$$

So 60 of those units come through B.

But only 40% of B's input came from A. Therefore, the amount attributable to A is

$$
60\times0.40=24.
$$

Thus,

$$
\frac{24}{100}=0.24.
$$

This gives us a useful rule:

$$
\boxed{\text{For connections along a path, multiply the connection strengths.}}
$$

## Representing direct connections with a matrix

<div style="text-align: center;">

:::{figure} ../static/effective_connectivity.png
:width: 1000px

**Effective connectivity Schematic.** A) Is a toy circuit with neurons a, b, c (left) and its representaiton in matrix form (right). B) is showing the matrix multiplication from neuron a to a. Adapted from [Yijie Yin, 2024](https://connectome-interpreter.readthedocs.io/en/latest/tutorials/matmul.html).
:::

</div>

Doing these calculations one neuron at a time would quickly become impractical. Instead, we can represent all the direct connections in a **matrix**.

For the toy circuit in the figure above (A, left), the connectivity matrix is shown (B, right)

The **rows** represent the source, or presynaptic, neurons:

$$
\text{row}=\text{{pre}}
$$

and the **columns** represent the target, or postsynaptic, neurons:

$$
\text{column}=\text{{post}}.
$$
Therefore, an entry {math}`M_{ij}` means

$$
\boxed{M_{ij}=\text{direct connection from neuron }i\text{ to neuron }j.}
$$

For example,

$$
M_{ab}=0.3
$$

in the matrix means:

$$
a\rightarrow b=0.3.
$$

The first row of the matrix is

$$
[0,\;0.3,\;0.5].
$$

This tells us all the direct connections leaving neuron \(a\):

$$
a\rightarrow a=0,
\qquad
a\rightarrow b=0.3,
\qquad
a\rightarrow c=0.5.
$$

Notice that this matrix contains only **direct, one-hop connections**.

So:

$$
\boxed{M=\text{connectivity after exactly one hop}}
$$

### From one hop to two hops

What if we want to know how strongly \(a\) can influence another neuron after going through **one intermediate neuron**?

That requires two connections (two arrows on the toy circuit), or **two hops**:

$$
a\rightarrow ?\rightarrow \text{target}.
$$

Let's say I want to know how strongly \(a\) can reach \(a\) in exactly two hops?

There are three possible intermediate neurons:
$$
a\rightarrow a\rightarrow a,
\qquad
a\rightarrow b\rightarrow a,
\qquad
a\rightarrow c\rightarrow a
$$

We calculate the strength of each path separately.

Through \(a\):

$$
a\xrightarrow{0}a\xrightarrow{0}a
$$

so

$$
0 \times 0 = 0.
$$

You might be wondering: how can {math}`a \rightarrow a \rightarrow a` connectivity exist? Remember, theoretically, our toy circuit could have self-connectivity. See the grey dashed lines:


<div style="text-align: center;">

:::{figure} ../static/effective_connectivity_annotate.png
:width: 1000px

**Effective connectivity Schematic.** A) Is a toy circuit with neurons a, b, c (left) and its representaiton in matrix form (right). B) is showing the matrix multiplication from neuron a to a. Adapted from [Yijie Yin, 2024](https://connectome-interpreter.readthedocs.io/en/latest/tutorials/matmul.html). The dashed grey lines in A denote hypothetical self-connectivity.
:::

</div>

You may notice that it is possible to have self-connecctions. Out circuit just doesn't have any! 

Through \(b\):

$$
a\xrightarrow{0.3}b\xrightarrow{0.2}a
$$

so

$$
0.3\times0.2=0.06.
$$

Through \(c\):

$$
a\xrightarrow{0.5}c\xrightarrow{0.8}a
$$

so

$$
0.5\times0.8=0.40.
$$

Sincer there are **more than one possible path** from \(a\) back to \(a\). We therefore add the contributions from all possible paths:

$$
0+0.06+0.40=0.46.
$$

This gives us our second rule:

$$
\boxed{\text{Multiply along a path; add across different paths.}}
$$

### Where does matrix multiplication come in?

Matrix multiplication performs exactly the calculation we just did.

To calculate effectivd connectivity of \(a,a\), we take **row \(a\)** from the first matrix:

$$
\begin{bmatrix}
0&0.3&0.5
\end{bmatrix}
$$

and **column \(a\)** from the second matrix:

$$
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}
$$

We multiply corresponding entries and add them:

$$
(M^2)_{aa}
=
\begin{bmatrix}
0&0.3&0.5
\end{bmatrix}
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}
$$

$$
=0(0)+0.3(0.2)+0.5(0.8)
$$

$$
=0+0.06+0.40
$$

$$
=\boxed{0.46}.
$$

Each multiplication corresponds to **one possible path**:

$$
\underbrace{0(0)}_{a\rightarrow a\rightarrow a}
+
\underbrace{0.3(0.2)}_{a\rightarrow b\rightarrow a}
+
\underbrace{0.5(0.8)}_{a\rightarrow c\rightarrow a}.
$$

This is why matrix multiplication is so useful for connectivity analysis. It automatically finds the contribution of every possible intermediate neuron.

For a real connectome, there may be thousands or millions of possible paths. Rather than calculating them individually, matrix multiplication performs all of these calculations simultaneously.

### Why do the columns sum to 1?

There is one final property of this particular connectivity matrix that is important.

Remember that a **column represents a target neuron**.

For example, column \(a\) contains

$$
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}.
$$

This means that neuron \(a\) receives:

$$
0\% \text{ from }a,
\qquad
20\% \text{ from }b,
\qquad
80\% \text{ from }c.
$$

Together,

$$
0+0.2+0.8=1.
$$

Or, in percentages,

$$
0\%+20\%+80\%=100\%.
$$

The same is true for every column:

$$
\boxed{\text{Each column sums to 1 because it represents 100\% of a target neuron's input.}}
$$

This also gives a useful way to remember the orientation of the matrix:

$$
\boxed{\text{rows = where the signal comes from/ source}}
$$

$$
\boxed{\text{columns = where the signal goes to / target}}
$$


### 2. Download the male CNS connectivity data

Choose a directory where you would like to store the data. Run the command below on terminal. Replace `YOUR DIRECTORY` below with that path:

```bash
wget -P 'YOUR DIRECTORY' https://github.com/YijieYin/connectome_data_prep/raw/refs/heads/main/data/maleCNS/mcns_inprop_all_neuron.npz
```

For example:

```bash
wget -P project_dir https://github.com/YijieYin/connectome_data_prep/raw/refs/heads/main/data/maleCNS/mcns_inprop_all_neuron.npz
```

The downloaded file is called `mcns_inprop_all_neuron.npz`.

### What is this file?

This file contains the connectivity matrix for the male CNS. In particular:

* `*_inprop` — input-normalised connectivity. Each entry represents the fraction of a postsynaptic neuron's input that comes from a given presynaptic neuron.

This is the connectivity matrix we will use to calculate **effective connectivity** in this tutorial.


Note: in this tutorial we are focusing on the malecns connectome. If you are interested, you are encouraged to explore the other connectomes that The Connectome Interpreter supports! 


In [15]:
mcns_inprop = sp.sparse.load_npz(project_dir + '/mcns_inprop_all_neuron.npz')
mcns_inprop

<Compressed Sparse Column sparse matrix of dtype 'float32'
	with 25083972 stored elements and shape (161429, 161429)>

# Wait ... Why can't I see my dataframe?

You may be confused by this output: *Compressed Sparse Column sparse matrix of dtype 'float32' with 25083972 stored elements and shape (161429, 161429)*

Please refer to this section of the [documentation](https://connectome-interpreter.readthedocs.io/en/latest/your_own_data.html#why-does-it-need-to-be-sparse) for a more detailed explanation. 

Essentially, our connectome is mostly sparse (most neurons don't connect to each other). Instead of storing one gigantic memory-intensive connectome matrix (~10GB), we can store this information into something called an edge list (basically only the connections that actually exist are recorded with a pre-synaptic neuron, post-synaptic neuron, and the input proportion columns).

| pre | post | inprop |
| --- | ---- | ------ |
| 101 | 205  |   0.32 |
| 101 | 873  |   0.15 |
| 205 | 992  |   0.71 |
| 873 | 101  |   0.04 |


# What does float32 mean?

A **bit** is the smallest unit of computer information. It can be either:

* `0`
* `1`

So **1 bit = one 0 or 1**.

A **byte** is 8 bits:

`10110101` = 8 bits = 1 byte.

A **float** is a way of storing a number that can have a decimal, such as:

```text
0.5
3.14
0.001
25.75
```

The number after `float` tells you **how many bits the computer uses to store that number**.

For example:

* `float16` → 16 bits → 2 bytes
* `float32` → 32 bits → 4 bytes
* `float64` → 64 bits → 8 bytes

More bits generally means the computer can represent numbers **more precisely**, but it also uses **more memory**.

So:

```python
float32
```

"Store each decimal number using 32 little 0s and 1s (32 bits), which takes 4 bytes of memory."

For this connectome, if you have millions of connection weights, using `float32` instead of `float64` can cut the memory needed for those weights roughly **in half**.



# Loading metadata 

Since the data in our connectivity matrix is only connections between neurons. What if we have a biologival question to ask? Like cell types etc. For each neuron ID (bodyId) in your matrix you have you have information describing the neurons themselves. This is mcns_meta. 

If you run mcns_meta.[column_of_interest] you can pick a column of interest


In [20]:
# Import metadata
mcns_meta = pd.read_csv(
    'https://raw.githubusercontent.com/YijieYin/connectome_data_prep/refs/heads/main/data/maleCNS/mcns_all_neuron_meta.csv',
    index_col=0,
)
mcns_meta

/var/folders/2f/zb94ck9x7sv6fn_sq___fw280000gn/T/ipykernel_1892/4089168952.py:2: DtypeWarning: Columns (0: somaLocation) have mixed types. Specify dtype option on import or set low_memory=False.
  mcns_meta = pd.read_csv(


,assignedOlHex1,assignedOlHex2,bodyid,flywireType,group,instance,somaSide,statusLabel,superclass,type,...,exitNerve,receptorType,somaLocation,tosomaLocation,status,top_nt,cell_type,coords,idx,sign
0,NaN,NaN,10001,DNp01,10001.0,DNp01(GF)_R,R,Roughly traced,descending_neuron,DNp01,...,NaN,NaN,[37124 22258 36274],NaN,Traced,acetylcholine,DNp01,NaN,0,1
1,NaN,NaN,10002,OCG01d,10002.0,OCG01d_L,L,Roughly traced,visual_projection,OCG01d,...,NaN,NaN,[50214 10279 27497],NaN,Traced,acetylcholine,OCG01d,NaN,1,1
2,NaN,NaN,10003,VCH,10003.0,VCH_R,R,Prelim Roughly traced,visual_centrifugal,VCH,...,NaN,NaN,[46716 21029 15714],NaN,Traced,gaba,VCH,NaN,2,-1
3,NaN,NaN,10005,AOTU019,10005.0,AOTU019_R,R,Roughly traced,cb_intrinsic,AOTU019,...,NaN,NaN,[41190 18491 12885],NaN,Traced,gaba,AOTU019,NaN,3,-1
4,NaN,NaN,10006,"VS1,VS2,VS3,VS4,VS5,VS6,VS7,VS8",10006.0,VS_L,L,Prelim Roughly traced,visual_projection,VS,...,NaN,NaN,[69604 28368 42459],NaN,Traced,acetylcholine,VS,NaN,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210460,NaN,NaN,1349927540,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNxx33,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNxx33,NaN,161424,1
210671,NaN,NaN,1387178136,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNpp45,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNpp45,NaN,161425,1
210739,NaN,NaN,1399810824,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNta34,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNta34,NaN,161426,1
210918,NaN,NaN,1434599043,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNta31,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNta31,NaN,161427,1


In [ ]:
mcns_meta.type #cell type

0           DNp01
1          OCG01d
2             VCH
3         AOTU019
4              VS
           ...   
210460     SNxx33
210671     SNpp45
210739     SNta34
210918     SNta31
211096     SNpp45
Name: type, Length: 161429, dtype: str

In [ ]:
mcns_meta.idx # bodyId

0              0
1              1
2              2
3              3
4              4
           ...  
210460    161424
210671    161425
210739    161426
210918    161427
211096    161428
Name: idx, Length: 161429, dtype: int64

In [ ]:
# Use type_to_function (which matched cell types with experimentally knwon fucntion we loaded at the top of notebook) to map to each cell type.
mcns_meta['known_function'] = mcns_meta.cell_type.map(type_to_function).fillna('')
mcns_meta.known_function

0                                      DNp01 escape_takeoff
1                                                          
2                                                          
3         AOTU019 frontal visual field object detection;...
4                                          VS Vertical flow
                                ...                        
210460                                                     
210671                                                     
210739                                                     
210918                                                     
211096                                                     
Name: known_function, Length: 161429, dtype: str

## Building lookup dictionaries

**Why do we use lookup dictionaries?**
Our connectome data uses an ID for each individual neuron. A lookup dictionary helps us connect that ID to useful biological information, such as its cell type, side, function, or location. 

For example:
```text
neuron ID → cell type
101       → L2
102       → L2
103       → Tm1
```

This information we need is already available in the metadata table `mcns_meta`, however, since most analysis functions in connectome_interpreter use these dictionaries we will create them. This dictionary set-up allows `connectome_interpreter` to take connections between individual neurons and **group them into categories**. For example, instead of looking at `101 → 103`, we can summarize it as **L2 → Tm1**.

Think of a lookup dictionary as a **quick translation table: neuron ID → biological information**. 

In [41]:
mcns_idx_to_type = dict(zip(mcns_meta.idx, mcns_meta.cell_type)) # bodyid assigned to a type
mcns_idx_to_sign = dict(zip(mcns_meta.idx, mcns_meta.sign)) # bodyid assigned to a neurotransmitter sign (1: excitatory -1: inhibitory)
mcns_idx_to_function = dict(zip(mcns_meta.idx, mcns_meta.known_function)) # bodyid assigned to a cell type function (if known)
mcns_type_to_sign = dict(zip(mcns_meta.cell_type, mcns_meta.sign)) # cell type assigned to a neurotransmitter sign (1: excitatory -1: inhibitory)

In [40]:
mcns_type_to_sign

{'DNp01': 1,
 'OCG01d': 1,
 'VCH': -1,
 'AOTU019': -1,
 'VS': 1,
 'CT1': -1,
 'OA-AL2i1': 1,
 'MBON01': -1,
 'HSN': 1,
 'HSE': 1,
 'AMMC-A1': 1,
 'LAL138': -1,
 'H2': 1,
 'CB2049': 1,
 'HSS': 1,
 'DNp18': 1,
 'LoVC16': -1,
 'pIP1': 1,
 'AOTU041': -1,
 'GNG116': -1,
 'DNg56': -1,
 'AstA1': -1,
 'VL2a_adPN': 1,
 'Dm17': -1,
 'OA-VUMa1': 1,
 'vCal2': -1,
 'ER5': -1,
 'SMP108': 1,
 'DNg100': 1,
 'MeVC11': 1,
 'MNx02': 1,
 'VA7m_lPN': 1,
 '5-HTPMPV03': 1,
 'Li39': -1,
 'LoVC14': -1,
 'DNpe013': 1,
 'mALD1': -1,
 'DNp20': 1,
 'MeVPOL1': 1,
 'DCH': -1,
 'DNp27': 1,
 'DNb05': 1,
 'LPi21': -1,
 'MeVPLp1': 1,
 'DNp30': -1,
 'AN07B004': 1,
 'AN12B011': -1,
 'PVLP010': -1,
 'DA1_lPN': 1,
 'OCG01b': 1,
 'dCal1': -1,
 'MeVC1': 1,
 'MBON06': -1,
 'OA-VUMa3': 1,
 'AVLP016': -1,
 'AVLP748m': 1,
 'DP1m_adPN': 1,
 'Am1': -1,
 'DH44': 1,
 'ANXXX033': 1,
 'PLP034': -1,
 'aSP22': 1,
 'DNpe042': 1,
 'aMe17a': 1,
 'AVLP597': -1,
 'lLN2F_a': 1,
 'oviIN': -1,
 'AVLP442': 1,
 'ExR6': -1,
 'LoVCLo3': 1,
 'PS100':